# Instacart Market Basket Prediction

## Overview

The goal of this project is to predict which products a customer is likely to purchase again in their next order.

I used the Instacart Market Basket dataset and focused on the customer-product relationship, looking at how often, how regularly, and how recently a customer has purchased a product.

The project combines MySQL for the initial data preparation and Python for feature engineering and modelling. I compared a few different models and evaluated the final predictions using precision, recall and F1, rather than relying on accuracy alone.


In [1]:
import pandas as pd 
import sklearn

In [30]:
df_prior = pd.read_parquet('data/prior.parquet')
df_train = pd.read_parquet('data/df_train.parquet')

## Dataset

The project uses the Instacart Market Basket dataset, which contains information about customers, orders, and the products in those orders.

The main tables used in this project are:

* `orders` — information about each customer's orders
* `products` — product information
* `aisles` and `departments` — product categories
* `order_products__prior` — products purchased in previous orders
* `order_products__train` — products from the order used for evaluation

I first used MySQL to join and prepare the data, then exported the resulting datasets to Parquet for use in Python.


## Baseline: Most Frequently Purchased Products

As a simple baseline, I predicted each customer's 10 most frequently purchased products from their previous order history.

This gives us a simple reference point to see whether the machine learning models are actually adding value beyond purchase frequency alone.


In [3]:
from sklearn.metrics import precision_score, recall_score, f1_score, accuracy_score

pred = df_prior.groupby(['user_id','product_id']).size().reset_index(name='freq')
pred = pred.sort_values(['user_id','freq'], ascending=[True, False])
pred = pred.groupby('user_id').head(10)
pred['predicted'] = 1


actual = df_train[['user_id','product_id']].copy()
actual['actual'] = 1


merged = pd.merge(pred[['user_id','product_id','predicted']], 
                  actual, 
                  on=['user_id','product_id'], 
                  how='outer').fillna(0)



print(f"Precision: {precision_score(merged['actual'], merged['predicted']):.3f}")
print(f"Recall:    {recall_score(merged['actual'], merged['predicted']):.3f}")
print(f"F1:        {f1_score(merged['actual'], merged['predicted']):.3f}")


Precision: 0.174
Recall:    0.253
F1:        0.206


The baseline gives us a starting point for comparison. Since this is an imbalanced problem, I will focus mainly on precision, recall and F1 when comparing it with the machine learning models.


## Feature Engineering

The main features are based on the customer's previous purchase behaviour for each product.

I wanted to capture three things:

* how often the customer has purchased the product
* how regularly they purchase it
* how recently they purchased it

Before creating the features, I split the history so that only orders before the target order are used. This keeps the feature calculation separate from the order we are trying to predict.


In [4]:
df_prior_new = df_prior.copy()

In [5]:
max_order = (
    df_prior_new
    .groupby('user_id')['order_number']
    .max()
    .rename('max_order')
    .reset_index())

In [6]:
df_prior_split = df_prior_new.merge(
    max_order,
    on='user_id',
    how='left')

In [7]:
df_history = df_prior_split[
    df_prior_split['order_number'] < df_prior_split['max_order']].copy()

In [8]:
df_target = df_prior_split[
    df_prior_split['order_number'] == df_prior_split['max_order']].copy()

### Creating the Historical Training Data

For each customer, I use their earlier orders as the history and their most recent order as the target.

The features are calculated using only the historical orders. The target is then whether each previously purchased product appears in the target order.

This gives the model the same setup it will have when making a real prediction: use what the customer has purchased so far to predict what they will purchase next.


In [9]:
def product_features(df):
    # Product-level history
    product_history = (
        df
        .groupby(['user_id', 'product_id'])['order_number']
        .agg(
            first_product_order='min',
            last_product_order='max',
            total_product_purchases='count').reset_index())

    # Maximum order available for each customer
    user_max_order = (
        df
        .groupby('user_id')['order_number']
        .max()
        .rename('max_user_order')
        .reset_index())

    product_history = product_history.merge(
        user_max_order,
        on='user_id',
        how='left')

    # Reorder rate
    product_history['reorder_rate'] = (
        (product_history['total_product_purchases'] - 1)
        / (
            product_history['max_user_order']
            - product_history['first_product_order'])).fillna(0)

    # Reorder intervals
    df_sorted = df.sort_values(
        ['user_id', 'product_id', 'order_number'])

    df_sorted['reorder_interval'] = (
        df_sorted
        .groupby(['user_id', 'product_id'])['order_number']
        .diff())

    avg_interval = (
        df_sorted
        .groupby(['user_id', 'product_id'])['reorder_interval']
        .mean()
        .reset_index(name='avg_reorder_interval'))

    product_history = product_history.merge(
        avg_interval,
        on=['user_id', 'product_id'],
        how='left')

    # Missing interval = product purchased only once
    product_history['interval_missing'] = (
        product_history['avg_reorder_interval']
        .isna()
        .astype(int))

    # Use 0 to represent no previous reorder interval
    product_history['avg_reorder_interval'] = (
        product_history['avg_reorder_interval']
        .fillna(0))

    # Recency
    product_history['recency'] = (
        product_history['max_user_order']
        - product_history['last_product_order'])

    return product_history

### Customer-Product Features

I used four features to describe the customer's previous behaviour with each product:

* **`reorder_rate`** — how often the customer has purchased the product again after the first purchase.
* **`avg_reorder_interval`** — the average number of orders between purchases of the product.
* **`interval_missing`** — indicates that there is no reorder interval because the product has only been purchased once.
* **`recency`** — the number of orders since the customer last purchased the product.

For `avg_reorder_interval`, missing values are filled with 0. The `interval_missing` feature keeps the distinction between a product with no previous reorder and one with an actual interval of 0.


In [10]:
product_history = product_features(df_history)

### Target Construction

The target is whether a product from the customer's previous purchase history appears in their next order.

I create the target by taking the products in the target order and marking them as `1`. Products from the customer's history that do not appear in that order are given a target of `0`.

This means the model is learning:

**customer + product history → purchased in the next order or not**


In [11]:
target_products = (
    df_target[['user_id', 'product_id']]
    .drop_duplicates()
    .assign(target=1))

In [12]:
product_history = product_history.merge(
    target_products,
    on=['user_id', 'product_id'],
    how='left')

In [13]:
product_history['target'] = (
    product_history['target']
    .fillna(0)
    .astype(int))

In [14]:
product_history['target'].value_counts(normalize=True)

target
0    0.899008
1    0.100992
Name: proportion, dtype: float64

### Final Evaluation Setup

After building the training data, I create a separate evaluation dataset using the full prior order history.

Here, the features are calculated using all available prior orders for each customer. The actual next order from `df_train` is then used to create the evaluation target.

This keeps the evaluation separate from the training data and tests how well the final model can predict the customer's next order using the information that would actually be available at prediction time.


In [15]:
product_history_eval = product_features(df_prior_new)

In [16]:
actual_next = df_train[['user_id', 'product_id']].drop_duplicates()

product_history_eval = product_history_eval.merge(
    actual_next.assign(target=1),
    on=['user_id', 'product_id'],
    how='left')

product_history_eval['target'] = (
    product_history_eval['target']
    .fillna(0)
    .astype(int))

### Final Evaluation Features

In [17]:
features = [
    'reorder_rate',
    'avg_reorder_interval',
    'interval_missing',
    'recency']

X_eval = product_history_eval[features]
y_eval = product_history_eval['target']

## Model Development

I started with Logistic Regression as a simple model that would give me an easy-to-understand baseline for the features.

I then compared it with Random Forest and XGBoost to see whether more flexible models could capture relationships that Logistic Regression might miss.

The models were evaluated using precision, recall and F1 at different prediction thresholds.


In [18]:
features = [
    'reorder_rate',
    'avg_reorder_interval',
    'interval_missing',
    'recency']

X = product_history[features]
y = product_history['target']

In [19]:
print(y_eval.value_counts())
print(f"Positive rate: {y_eval.mean():.4f}")

target
0    12479129
1      828824
Name: count, dtype: int64
Positive rate: 0.0623


In [20]:
from sklearn.linear_model import LogisticRegression

model = LogisticRegression(
    max_iter=100,)

model.fit(X, y);

In [21]:
coefficients = pd.DataFrame({
    'feature': features,
    'coefficient': model.coef_[0]})

coefficients

,feature,coefficient
0,reorder_rate,2.088885
1,avg_reorder_interval,-0.021047
2,interval_missing,-0.472423
3,recency,-0.123233


In [22]:
from sklearn.metrics import precision_score, recall_score, f1_score

y_eval_proba = model.predict_proba(X_eval)[:, 1]

y_eval_pred = (y_eval_proba >= 0.5).astype(int)

precision = precision_score(y_eval, y_eval_pred)
recall = recall_score(y_eval, y_eval_pred)

print(f"Precision: {precision:.4f}")
print(f"Recall:    {recall:.4f}")

Precision: 0.3200
Recall:    0.1208


In [23]:
for threshold in [0.4, 0.3]:
    y_pred = (y_eval_proba >= threshold).astype(int)

    precision = precision_score(y_eval, y_pred)
    recall = recall_score(y_eval, y_pred)
    f1 = f1_score(y_eval, y_pred)

    print(f"Threshold: {threshold}")
    print(f"Precision: {precision:.4f}")
    print(f"Recall:    {recall:.4f}")
    print(f"F1:        {f1:.4f}")
    print()

Threshold: 0.4
Precision: 0.3281
Recall:    0.2075
F1:        0.2542

Threshold: 0.3
Precision: 0.2781
Recall:    0.3377
F1:        0.3050



### Comparing Models

I then tested Random Forest and XGBoost using the same features and evaluation setup.

The goal was not to keep adding complexity, but to see whether the more flexible models could improve the results compared with Logistic Regression.


In [24]:
from sklearn.ensemble import RandomForestClassifier

rf_model = RandomForestClassifier(
    n_estimators=200,
    max_depth=10,
    n_jobs=-1,
    random_state=42,
    class_weight=None)

rf_model.fit(X, y);

In [25]:
y_eval_proba_rf = rf_model.predict_proba(X_eval)[:, 1]

In [26]:
for threshold in [0.5, 0.4, 0.3]:
    y_pred = (y_eval_proba_rf >= threshold).astype(int)

    precision = precision_score(y_eval, y_pred)
    recall = recall_score(y_eval, y_pred)
    f1 = f1_score(y_eval, y_pred)

    print(f"Threshold: {threshold}")
    print(f"Precision: {precision:.4f}")
    print(f"Recall:    {recall:.4f}")
    print(f"F1:        {f1:.4f}")
    print()

Threshold: 0.5
Precision: 0.3936
Recall:    0.0957
F1:        0.1539

Threshold: 0.4
Precision: 0.3241
Recall:    0.2694
F1:        0.2942

Threshold: 0.3
Precision: 0.2834
Recall:    0.3771
F1:        0.3236



In [27]:
from xgboost import XGBClassifier

xgb_model = XGBClassifier(
    n_estimators=200,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    n_jobs=-1,
    eval_metric='logloss')

xgb_model.fit(X, y);

In [28]:
y_eval_proba_xgb = xgb_model.predict_proba(X_eval)[:, 1]

In [29]:
for threshold in [0.5, 0.4, 0.3, 0.25]:
    y_pred = (y_eval_proba_xgb >= threshold).astype(int)

    precision = precision_score(y_eval, y_pred)
    recall = recall_score(y_eval, y_pred)
    f1 = f1_score(y_eval, y_pred)

    print(f"Threshold: {threshold}")
    print(f"Precision: {precision:.4f}")
    print(f"Recall:    {recall:.4f}")
    print(f"F1:        {f1:.4f}")
    print()

Threshold: 0.5
Precision: 0.3939
Recall:    0.0944
F1:        0.1523

Threshold: 0.4
Precision: 0.3252
Recall:    0.2647
F1:        0.2918

Threshold: 0.3
Precision: 0.2823
Recall:    0.3793
F1:        0.3237

Threshold: 0.25
Precision: 0.2624
Recall:    0.4299
F1:        0.3259



### Final Model and Threshold

I used the XGBoost model for the final predictions and selected a threshold of **0.25**.

The default threshold of 0.5 gave high precision but relatively low recall. Lowering the threshold increased recall, while precision decreased. At 0.25, the model achieved:

* **Precision:** 0.2624
* **Recall:** 0.4299
* **F1:** 0.3259

The F1 score was slightly higher at 0.25 than at the other thresholds tested, while also giving the model higher recall.

For this project, I chose 0.25 as the final decision threshold because predicting the products a customer is likely to reorder requires finding more of the relevant products rather than being overly conservative with predictions.


### Limitations

The model only uses a customer's previous relationship with a product to make its prediction. It does not consider things like product category, aisle, or whether a product is generally popular across customers. This was intentional, since I wanted to first understand how much could be predicted from the customer-product relationship itself.

Another limitation is that the model generates predictions for products the customer has purchased before. It therefore does not address the separate problem of recommending products that a customer has never purchased.

The final model also has a noticeable precision-recall trade-off. At the selected threshold, the model identifies a reasonable proportion of products that appear in the next order, but a significant number of its predicted reorders are still incorrect. This reflects the difficulty of predicting the exact contents of a customer's next basket from purchase history alone.

Finally, the model is built around historical order behaviour, so it does not capture changes in a customer's preferences over time beyond what can be inferred from their purchasing history.
